## Trabajo Final - Fundamentos de Inferencia Causal
### Efectos de la Pensión 65 en la Oferta Laboral de los Jóvenes (ENAHO 2024)

**Curso:** Fundamentos de Inferencia Causal

**Integrantes:**
- Adrian Salinas Alcántara
- Estefanía Apaza Díaz
- Maria Jesus Perez Zarate
- Fabiana Marcos


### **1. Introducción**

#### 1.1 Contextualización

El presente trabajo final evalúa el tema de los programas sociales y su impacto en la insercción laboral mediante el análisis del programa Pensión 65. Este otorga una transferencia bimensual de S/250 a adultos mayores que superan los 65 años y residen en hogares en situación de pobreza extrema, según la clasificación del SISFOH (Ministerio de Desarollo e Inclusión Social, s.f.).

La evidencia sobre los efectos directos de este tipo de transferencias no contributivas en América Latina es abundante. Los beneficiarios tienden a reducir su participación laboral, mejoran modestamente sus indicadores de salud y aumentan su seguridad alimentaria. Pero los efectos indirectos sobre otros miembros del hogar han sido poco estudiados en el Perú.

#### 1.2 Pregunta de investigación y objetivo

La pregunta que se busca responder mediante esta investigación es la siguiente: ¿El que un adulto mayor reciba Pensión 65 reduce la necesidad de que los jóvenes (14-24 años) que cohabitan con ellos entren al mercado laboral?

Este trabajo aborda precisamente ese vacío, centrándose en la oferta laboral de los jóvenes de 14 a 24 años que conviven con el adulto mayor beneficiario.

#### 1.3 Mecanismo

El mecanismo detrás de este posible efecto de derrame es directo. Se espera que un ingreso adicional no condicionado releje la restricción presupuestaria del hogar. Así, la familia deja de depender tanto de que todos sus miembros trabajen para cubrir necesidades básicas.

En ese contexto, un joven que de otro modo se habría insertado prematuramente en empleos informales o agrícolas podría postergar esa decisión. Podría dedicar más tiempo a estudiar o simplemente no verse forzado a trabajar. Es un efecto ingreso puro que opera a nivel del hogar y que altera las decisiones de personas que no son beneficiarias directas del programa.

### **2. Estrategia de evaluación del efecto causal**

#### 2.1 Diseño

La estrategia de identificación se basa en un diseño de Regresión Discontinua. Como Pensión 65 exige que el adulto mayor tenga 65 años cumplidos para acceder al beneficio, etsa regla genera una discontinuidad clara. Un hogar donde el adulto mayor tiene 64 años es casi idéntico a uno donde acaba de cumplir 65, tanto en características observables como no observables. La diferencia es que solo el segundo grupo califica.

La variable de asignación es la edad del adulto mayor centrada en 65, con un ancho de banda de 55 a 75 años. Dado que no todos los elegibles reciben la pensión de inmediato, la estimación corresponde a un efecto de Intención de Tratamiento en forma reducida.

#### 2.2 Filtros a la base de datos

Se filtrará el total de observaciones por aquellos hogares con adultos mayores beneficiarios del programa Pensión 65, el cual requiere que el adulto mayor tenga al menos 65 años cumplidos. Asimismo, se aplicará un segundo filtro para encontrar aquellas observaciones en las que hay jóvenes de 14 a 24 años en el hogar.

Posteriormente, se utilizará la edad del adulto mayor del hogar como variable de asignación (Running Variable). Además, se compararán a estos jóvenes en hogares de pobreza extrema donde el adulto mayor tiene entre 60 y 64 años (Control: no ) contra aquellos donde el adulto mayor tiene entre 65 y 69 años (Tratados: sí califica). Al estar muy cerca del umbral de edad, ambos hogares sufren choques idénticos, siendo su única diferencia la recepción exógena de este bono.

#### 2.3 Amenazas a la validez interna

Existen dos amenazas a la validez interna de este diseño. Primero, la manipulación de la variable de asignación. Si las personas pudieran alterar su edad registrada para acceder al programa, el supuesto de continuidad se rompería. Esto no ocurre en el Perú. La elegibilidad se verifica cruzando datos con RENIEC y no hay evidencia de falsificación sistemática.

Segundo, a los 65 años también se activa la posibilidad de jubilarse bajo el sistema de ONP. Si eso ocurriera simultáneamente, el efecto capturaría ambos programas. Para evitar esa confusión, la muestra se restringe a hogares pobres y pobres extremos, donde la probabilidad de haber cotizado en un sistema contributivo formal es prácticamente nula.

### **3. Base de datos y descripción de las variables**

Para el análisis propuesto, se ha elegido la base de datos de la Encuesta Nacional de Hogares (ENAHO) 2024 del Instituto Nacional de Estadística e Informática (INEI). Específicamente, se ha construido una base a nivel individual a partir de los siguientes módulos:
- 500 - Empleo
- 200 - Características de los miembros del hogar
- Sumaria

 Las variables seleccionadas de estos módulos son las siguientes:

 **Variable dependiente**
 1. *trabaja* (dummy)

```
1: El joven trabajó al menos 1 hora la semana pasada
0: El joven no trabajó la semana pasada
```

**Variables independientes**

1. *edad_abuelo*

Variable de asignación. Edad del adulto mayor con relación a los 65 años.

2. *tratamiento*

Nos concentramos en si su edad cumple el requisito de tener igual o más de 65 años para ser beneficiarios de la política pública de Pensión 65.

```
1: Edad del adulto mayor => 65
0: Edad del adulto mayor entre 60 y 64
```
**Controles**

- Sexo
- Edad del joven
- Área rural

#### Preparación de la base de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings("ignore")

def cleanup_enaho(df):
    df.columns = [c.strip().upper() for c in df.columns]
    df.columns = ["AÑO" if ("A" in c and "O" in c and len(c)==2) else c for c in df.columns]
    return df

Usamos las columnas específicas del Módulo 200 (Características), Módulo 500 (Empleo) y Módulo Sumaria (Pobreza) de la ENAHO 2024.

In [ ]:
# Módulo 200 (Características y Edades)
# P203 = Parentesco, P207 = Sexo, P208A = Edad, P209 = Estado Civil
df200 = cleanup_enaho(pd.read_csv("Enaho01-2024-200.csv", encoding="latin-1", low_memory=False,
                                  usecols=["CONGLOME","VIVIENDA","HOGAR","CODPERSO","P207","P208A", "P203", "P209"]))
df200["edad"] = pd.to_numeric(df200["P208A"], errors="coerce")
df200["mujer"] = np.where(pd.to_numeric(df200["P207"], errors="coerce") == 2, 1, 0)

# Módulo 500 (Empleo - Jóvenes)
# P501 = ¿La semana pasada trabajó al menos 1 hora?
df500 = cleanup_enaho(pd.read_csv("Enaho01a-2024-500.csv", encoding="latin-1", low_memory=False,
                                  usecols=["CONGLOME","VIVIENDA","HOGAR","CODPERSO","P501", "P507"]))
df500["trabaja"] = np.where(pd.to_numeric(df500["P501"], errors="coerce") == 1, 1, 0) # 1=Sí trabajó

# Módulo Sumaria (Niveles de Pobreza y Geografía)
dfsum = cleanup_enaho(pd.read_csv("Sumaria-2024.csv", encoding="latin-1", low_memory=False,
                                  usecols=["CONGLOME","VIVIENDA","HOGAR","POBREZA", "ESTRSOCIAL", "DOMINIO"]))
dfsum["rural"] = np.where(dfsum["DOMINIO"] == 8, 1, 0)

FileNotFoundError: [Errno 2] No such file or directory: 'Enaho01-2024-200.csv'

Construcción de la base de datos de hogares y configuración del diseño RDD

In [ ]:
# Identificamos la edad del Adulto Mayor Principal del hogar (Jefe o Cónyuge)
abuelos = df200[(df200["P203"].isin([1, 2])) & (df200["edad"] >= 55)].copy()
abuelos["edad"] = pd.to_numeric(abuelos["edad"])
hogar_abuelos = abuelos.groupby(["CONGLOME","VIVIENDA","HOGAR"])["edad"].max().reset_index()
hogar_abuelos.rename(columns={"edad": "edad_abuelo"}, inplace=True)

# Identificamos a los Jóvenes (14 a 24 años) en el hogar
jovenes = df200[(df200["edad"] >= 14) & (df200["edad"] <= 24)].copy()

# Seleccionamos las variables identificadoras y de trabajo de los jóvenes
df_ind = pd.merge(jovenes, df500, on=["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"], how="inner")

# Unimos la información de los jóvenes con la edad del adulto mayor del hogar
df_analisis = pd.merge(df_ind, hogar_abuelos, on=["CONGLOME", "VIVIENDA", "HOGAR"], how="inner")

# Unimos con pobreza y geografía del hogar
df_analisis = pd.merge(df_analisis, dfsum, on=["CONGLOME", "VIVIENDA", "HOGAR"], how="inner")

# Filtro de vivienda: Hogares Pobres/Extremos y ancho de banda [55 a 75 años del abuelo]
df_rdd = df_analisis[(df_analisis["POBREZA"].isin([1, 2])) &
                     (df_analisis["edad_abuelo"] >= 55) &
                     (df_analisis["edad_abuelo"] <= 75)].copy()

# Centramos la edad en el corte de 65 años (Pensión 65)
df_rdd["edad_centrada"] = df_rdd["edad_abuelo"] - 65
df_rdd["tratamiento"] = np.where(df_rdd["edad_centrada"] >= 0, 1, 0)

print(f"Observaciones finales (jóvenes): {len(df_rdd).")

### **4. Análisis Esploratorio**
Revisamos medias de variables de control por Tratamiento para confirmar que no hay diferencias ex-ante (Balance Test).

#### 4.1 Tablas descriptivas


In [ ]:
desc = df_rdd.groupby("tratamiento")[["edad", "mujer", "rural"]].mean().reset_index()
desc.rename(columns={"tratamiento": "Recibe Pensión 65 (Abuelo>=65)",
                     "edad": "Edad Promedio Joven",
                     "mujer": "% Mujeres",
                     "rural": "% Zona Rural"}, inplace=True)

desc["Recibe Pensión 65 (Abuelo>=65)"] = desc["Recibe Pensión 65 (Abuelo>=65)"].map({0: "Control (No Recibe)", 1: "Tratado (Sí Recibe)"})
print("=== ESTADÍSTICAS DESCRIPTIVAS DE LOS JÓVENES ===")
display(desc)

#### 4.2 Gráfico Descriptivo y RDD Salto

Mostramos cómo cambia la tendencia de trabajo en la edad de corte (Salto Discontinuo).

In [ ]:
plt.figure(figsize=(10, 6))
# Lado Izquierdo (Control)
sns.regplot(x="edad_centrada", y="trabaja", data=df_rdd[df_rdd["edad_centrada"] < 0],
            scatter_kws={"alpha":0.3, "color":"gray"}, line_kws={"color":"red", "lw":2},
            order=1, ci=95, label="Control (Abuelo < 65)")
# Lado Derecho (Tratados)
sns.regplot(x="edad_centrada", y="trabaja", data=df_rdd[df_rdd["edad_centrada"] >= 0],
            scatter_kws={"alpha":0.3, "color":"blue"}, line_kws={"color":"green", "lw":2},
            order=1, ci=95, label="Tratados (Abuelo >= 65)")

plt.axvline(0, color="black", linestyle="--", label="Corte Pensión 65 (65 años)")
plt.title("Discontinuidad: Efecto de Pensión 65 de Abuelos en el Trabajo de Jóvenes")
plt.xlabel("Edad del Adulto Mayor (Centrada en 0 = 65 años)")
plt.ylabel("Probabilidad de que el Joven Trabaje")
plt.legend()
plt.show()

Este gráfico muestra cómo cambia la probabilidad de que el joven trabaje según la edad del adulto mayor. La línea vertical marca los 65 años, que es el punto de elegibilidad de Pensión 65. Observamos que la probabilidad de trabajar parece disminuir ligeramente después del umbral, lo que sugiere que el programa podría reducir la necesidad de que los jóvenes trabajen.

#### 4.3 Modelos OLS - Simple y Múltiple


In [ ]:
# Modelo 1: MCO Simple (Solo tratamiento)
ols_simple = smf.ols("trabaja ~ tratamiento", data=df_rdd).fit(cov_type="HC3")

# Modelo 2: MCO Múltiple RDD (Polinomio de edad centrada y controles)
formula_mco = "trabaja ~ tratamiento + edad_centrada + tratamiento:edad_centrada + edad + C(mujer) + C(rural)"
ols_multiple = smf.ols(formula_mco, data=df_rdd).fit(cov_type="HC3")

from statsmodels.iolib.summary2 import summary_col
print("\n=== RESULTADOS MÍNIMOS CUADRADOS ORDINARIOS (MCO) ===")
print(summary_col([ols_simple, ols_multiple], stars=True,
                  model_names=["MCO Simple", "MCO Múltiple (RDD)"],
                  info_dict={"N":lambda x: f"{int(x.nobs)}", "R2":lambda x: f"{x.rsquared:.3f}"}))

Los resultados OLS muestran que el coeficiente de tratamiento es negativo tanto en el modelo simple como en el modelo múltiple. En particular, en la especificación RDD cruzar el umbral de 65 años se asocia con una reducción de aproximadamente 8 puntos porcentuales en la probabilidad de que el joven trabaje. Sin embargo, este efecto no es estadísticamente significativo, por lo que debe interpretarse con cautela.

#### 4.4 Aplicación de Modelos No Lineales (Probit) y Marginales
Dado que `trabaja` es binario [0,1], re-estimamos el impacto exógeno usando un modelo PROBIT con Errores Estándar Robustos y extraemos los Efectos Marginales (AME).

In [ ]:
# Modelo Probit
probit_model = smf.probit(formula_mco, data=df_rdd).fit(cov_type="HC3", disp=0)

# Efectos Marginales Promedio
marginales = probit_model.get_margeff()

print("\n=== EFECTOS MARGINALES PROMEDIO (PROBIT) ===")
print(marginales.summary())

Cruzar el umbral de 65 años (Tratamiento=1) tiene un impacto marginal negativo en la probabilidad base del joven de trabajar. Así, la probabilidad de que el joven trabaje se reduce en aproximadamente en -0.0832 puntos, manteniendo constantes las demás variables.

No obstante, el coeficiente no es estadísticamente significativo, ya que el p-value es mayor a 0.05 (p = 0.137). Por ello, no es posible afirmar que el programa tenga un efecto causal sobre la probabilidad del joven de trabajar.

#### 4.5 Análisis de Heterogeneidad
Se evalúa si el bono Pensión 65 reduce más el trabajo en jóvenes que viven en áreas urbanas o rurales mediante el cruce de variables y la división de muestras.

In [ ]:
print("\n=== HETEROGENEIDAD POR ÁREA GEOGRÁFICA ===")
# Modelo Interaccionado (Controlando si el impacto cambia por ser Rural)
formula_het = "trabaja ~ tratamiento * C(rural) + edad_centrada + tratamiento:edad_centrada + edad + C(mujer)"
ols_het = smf.ols(formula_het, data=df_rdd).fit(cov_type="HC3")
print(ols_het.summary().tables[1])


=== HETEROGENEIDAD POR ÁREA GEOGRÁFICA ===
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -0.1181      0.091     -1.294      0.196      -0.297       0.061
C(rural)[T.1]                -0.1395      0.041     -3.418      0.001      -0.220      -0.060
C(mujer)[T.1]                -0.1207      0.028     -4.370      0.000      -0.175      -0.067
tratamiento                  -0.0722      0.056     -1.283      0.199      -0.182       0.038
tratamiento:C(rural)[T.1]    -0.0913      0.071     -1.277      0.202      -0.231       0.049
edad_centrada                 0.0005      0.006      0.081      0.935      -0.011       0.012
tratamiento:edad_centrada     0.0102      0.009      1.082      0.279      -0.008       0.029
edad                          0.0340      0.004      7.670      0.000       0.025       0.043


En el modelo OLS con interacciones, cruzar el umbral de 65 años (Tratamiento=1) se asocia en promedio a una reducción de 7.2 puntos porcentuales en la probabilidad de que el joven urbano trabaje (-0.0722), manteniendo constantes las demás variables.

Sin embargo, el coeficiente no es estadísticamente significativo (p = 0.199), por lo que no es posible afirmar que el programa tenga un efecto causal sobre la probabilidad del joven urbano de trabajar.

Asimismo, cruzar el umbral de 65 años (Tratamiento=1) se asocia en promedio a una reducción de 9.1 puntos porcentuales en la probabilidad de que el joven rural trabaje (-0.0913), manteniendo constantes las demás variables, lo que indicaría que el efecto del programa podría ser más pronunciado en hogares rurales.

Sin embargo, el coeficiente no es estadísticamente significativo (p = 0.202), por lo que no es posible afirmar que el programa tenga un efecto causal sobre la probabilidad del joven rural de trabajar.

In [ ]:
# Modelos Separados (Submuestras)
print("\n--- Impacto RDD en Jóvenes URBANOS ---")
ols_urb = smf.ols("trabaja ~ tratamiento + edad_centrada", data=df_rdd[df_rdd["rural"]==0]).fit(cov_type="HC3")
print(f"Coeficiente de Tratamiento Urbano: {ols_urb.params['tratamiento']:.3f} (P-value: {ols_urb.pvalues['tratamiento']:.3f})")

print("\n--- Impacto RDD en Jóvenes RURALES ---")
ols_rur = smf.ols("trabaja ~ tratamiento + edad_centrada", data=df_rdd[df_rdd["rural"]==1]).fit(cov_type="HC3")
print(f"Coeficiente de Tratamiento Rural: {ols_rur.params['tratamiento']:.3f} (P-value: {ols_rur.pvalues['tratamiento']:.3f})")


--- Impacto RDD en Jóvenes URBANOS ---
Coeficiente de Tratamiento Urbano: -0.096 (P-value: 0.132)

--- Impacto RDD en Jóvenes RURALES ---
Coeficiente de Tratamiento Rural: -0.154 (P-value: 0.226)


Para jóvenes urbanos, el efecto estimado del tratamiento es −0.096 (p = 0.132), mientras que para jóvenes rurales es −0.154 (p = 0.226). Aunque la magnitud del efecto parece mayor en zonas rurales, ninguno de los coeficientes es estadísticamente significativo.